# Generative AI 014 — Vector Stores

Chroma and FAISS are not installed here, but `InMemoryVectorStore` ships inside
`langchain-core` and implements the **same interface** — so the whole CRUD and
search surface runs with **no API key**.

| Part | What we check |
|---|---|
| A | keyword matching: **100%** on unrelated films, **25%** on similar ones |
| B | a real vector store — add, search, filter, delete, `as_retriever()` |
| C | the stand-in embedder's honest limit |
| D | indexing: **19.6×** faster, and **21%** vs **100%** recall |

Needs `langchain-core`, `scikit-learn`, `numpy`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.runnables import Runnable
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

## Part A — Why keyword matching cannot power a recommender

In [ ]:
MOVIES = {
    "My Name Is Khan":        dict(director="Karan Johar", lead="Shah Rukh Khan",
                                   genre="drama", year=2010),
    "Kabhi Alvida Naa Kehna": dict(director="Karan Johar", lead="Shah Rukh Khan",
                                   genre="drama", year=2006),
    "Taare Zameen Par":       dict(director="Aamir Khan", lead="Darsheel Safary",
                                   genre="drama", year=2007),
    "A Beautiful Mind":       dict(director="Ron Howard", lead="Russell Crowe",
                                   genre="drama", year=2001),
}

def keyword_score(a, b):
    A, B = MOVIES[a], MOVIES[b]
    return sum([A["director"] == B["director"], A["lead"] == B["lead"],
                A["genre"] == B["genre"], abs(A["year"] - B["year"]) <= 5]) / 4

fp = keyword_score("My Name Is Khan", "Kabhi Alvida Naa Kehna")
fn = keyword_score("Taare Zameen Par", "A Beautiful Mind")
print(f"same director/lead/genre/era, different stories : {fp:.0%}")
print(f"different everything, same theme                : {fn:.0%}")

assert fp == 1.0 and fn == 0.25
print()
print("A false positive at 100% and a false negative at 25%, from the same")
print("four fields. Both come from comparing labels instead of stories.")

## Part B — A vector store, end to end

In [ ]:
class TfidfEmbeddings(Embeddings):
    """A real Embeddings implementation that runs offline.
    It compares WORDS, not meaning - see Part C."""
    def __init__(self, corpus):
        self.v = TfidfVectorizer(stop_words="english").fit(corpus)
    def embed_documents(self, texts):
        return self.v.transform(texts).toarray().tolist()
    def embed_query(self, text):
        return self.v.transform([text]).toarray()[0].tolist()

PLAYERS = [
    ("Virat Kohli is a batter for RCB, famous for chasing targets in run chases.",
     {"team": "RCB", "role": "batter"}),
    ("Jasprit Bumrah is a fast bowler for MI, known for yorkers at the death.",
     {"team": "MI", "role": "bowler"}),
    ("MS Dhoni is a wicketkeeper and captain for CSK who finishes close matches.",
     {"team": "CSK", "role": "wicketkeeper"}),
    ("Ravindra Jadeja is a bowling all-rounder for CSK and an excellent fielder.",
     {"team": "CSK", "role": "all-rounder"}),
]
texts = [t for t, _ in PLAYERS]
store = InMemoryVectorStore(embedding=TfidfEmbeddings(texts))
ids = store.add_documents([Document(page_content=t, metadata=m) for t, m in PLAYERS])
print("added", len(ids), "documents")

In [ ]:
print("similarity_search('Who is a bowler?', k=2):")
for d in store.similarity_search("Who is a bowler?", k=2):
    print("   ", d.metadata["role"], "|", d.page_content[:46])

print("\nsimilarity_search_with_score('fast bowler yorkers', k=2):")
for d, s in store.similarity_search_with_score("fast bowler yorkers", k=2):
    print(f"    {s:.4f}  {d.page_content[:46]}")

print("\nmetadata filter - CSK only:")
csk = store.similarity_search("captain", k=5,
                              filter=lambda d: d.metadata.get("team") == "CSK")
print("   ", [d.metadata["role"] for d in csk])
assert {d.metadata["team"] for d in csk} == {"CSK"}

In [ ]:
store.delete(ids=[ids[0]])
print("after delete:", len(store.store), "documents")

retriever = store.as_retriever(search_kwargs={"k": 2})
print("as_retriever ->", type(retriever).__name__)
print("is a Runnable:", isinstance(retriever, Runnable))

assert isinstance(retriever, Runnable)
print()
print("THAT is why RAG is one line. A retriever is a Runnable, so")
print("   retriever | prompt | model | parser")
print("composes exactly like everything else in lessons 009-011.")

## Part C — The honest limit of the stand-in embedder

In [ ]:
store2 = InMemoryVectorStore(embedding=TfidfEmbeddings(texts))
store2.add_documents([Document(page_content=t, metadata=m) for t, m in PLAYERS])

print("a query sharing words with the documents:")
print("   ", store2.similarity_search("Who is a bowler?", k=1)[0].metadata["role"])

print("\na query meaning the same thing in different words:")
try:
    store2.similarity_search("Who keeps wicket?", k=2)
    print("    returned results")
except ValueError as e:
    print("    ValueError:", e)

print()
print("'wicketkeeper' is one token, so 'wicket' is not in the vocabulary.")
print("The query embeds to an ALL-ZERO vector, and the cosine of zero is")
print("undefined - so the store cannot rank anything and raises.")
print()
print("Lesson 003's result one step worse: there the paraphrase scored")
print("0.0000; here it cannot produce a score at all.")
print()
print("Everything in Part B shows the STORE works. None of it shows")
print("retrieval QUALITY - that is the embedding model's job.")

## Part D — Indexing: what the speedup costs

Searching a million vectors one at a time is O(n). The standard fix is to
cluster them and search only the winning cluster. The usual telling adds that
this "roughly preserves the best result" — which is testable.

In [ ]:
N, DIM, QN = 20_000, 64, 300
rng = np.random.default_rng(0)
unit = lambda a: a / np.linalg.norm(a, axis=1, keepdims=True)

def evaluate(data, queries, n_clusters):
    """Comparisons per query, and how often we find the TRUE nearest."""
    km = KMeans(n_clusters=n_clusters, n_init=3, random_state=0).fit(data)
    labels, centroids = km.labels_, km.cluster_centers_
    true_best = (queries @ data.T).argmax(axis=1)         # the brute-force answer
    hits, comps = 0, []
    for i, q in enumerate(queries):
        cluster = (centroids @ q).argmax()                # n_clusters comparisons
        members = np.where(labels == cluster)[0]          # then only this cell
        comps.append(n_clusters + len(members))
        if len(members) and members[(data[members] @ q).argmax()] == true_best[i]:
            hits += 1
    return float(np.mean(comps)), hits / len(queries)

data    = unit(rng.standard_normal((N, DIM)))
queries = unit(rng.standard_normal((QN, DIM)))

print(f"{'clusters':>9}{'comparisons':>13}{'speedup':>9}{'recall@1':>11}")
print(f"{'brute':>9}{N:>13,}{1.0:>8.1f}x{1.0:>10.0%}")
for nc in (5, 20, 50, 100):
    comps, recall = evaluate(data, queries, nc)
    print(f"{nc:>9}{comps:>13,.0f}{N/comps:>8.1f}x{recall:>10.1%}")

**19.6× cheaper at 20 clusters, and it finds the true nearest neighbour one
time in five.** That is not "roughly preserving the best result".

But the reason is the **data**, not the method. Those were uniformly random
vectors — no cluster structure for k-means to find. Real embeddings are not
random: text about cricket lands near other text about cricket.

In [ ]:
K = 20
centres = rng.standard_normal((K, DIM)) * 3
clustered   = unit(centres[rng.integers(0, K, N)] + rng.standard_normal((N, DIM)) * 0.5)
q_assign    = rng.integers(0, K, QN)
clustered_q = unit(centres[q_assign] + rng.standard_normal((QN, DIM)) * 0.5)

c_rand, r_rand = evaluate(data, queries, 20)
c_clus, r_clus = evaluate(clustered, clustered_q, 20)

print(f"{'data':<38}{'speedup':>9}{'recall@1':>11}")
print(f"{'random (no structure)':<38}{N/c_rand:>8.1f}x{r_rand:>10.1%}")
print(f"{'clustered (real structure)':<38}{N/c_clus:>8.1f}x{r_clus:>10.1%}")

assert r_rand < 0.4 and r_clus > 0.95
assert abs(c_rand - c_clus) / c_rand < 0.10      # the SAME speedup
print()
print("Same speedup. Recall 21% against 100%.")
print()
print("So the speedup is free and the accuracy is not. Whether an index costs")
print("you accuracy depends entirely on whether your vectors have structure.")
print("Embeddings usually do - which is why ANN search works in practice, and")
print("why a vector database reports recall as a tunable number rather than")
print("promising exact results.")

> **What that measured.** k-means over synthetic vectors, not a benchmark of
> any real index. Production systems use HNSW or IVF-PQ, which are far
> cleverer than one flat layer of centroids. The **dependence on structure**
> is the transferable part, not the numbers.

## What to take away

- Keyword matching fails **both ways**: 100% on unrelated, 25% on similar.
- A vector store gives storage with metadata, similarity search, indexing, CRUD.
- **One interface** — this code is Chroma's code apart from the constructor.
- **`as_retriever()` returns a `Runnable`**, so RAG composes with `|`.
- Indexing: **19.6× faster**, recall **21%** on random vs **100%** on clustered.

## Exercises

1. Sweep the cluster count on the *clustered* data. Does recall ever fall?
   At what point, and why?
2. Search the two nearest clusters instead of one. How much recall do you buy
   back on the random data, and what does it cost?
3. Replace `TfidfEmbeddings` with `DeterministicFakeEmbedding` from
   `langchain_core.embeddings`. Retrieval still "works" — why is that worse
   than the `ValueError` in Part C?
4. Part C raised on a zero vector. Add a guard to `TfidfEmbeddings` that
   detects it and raises something more useful. What should it say?
5. If you have an API key: swap in `OpenAIEmbeddings` and ask "Who keeps
   wicket?" again. That is the measurement this notebook could not make.